In [ ]:
import bentoml
import numpy as np

In this solution, we will focus mainly on the service code (to be stored in a service.py file or another name of your choice)


In [ ]:
# First, we define the list of regions within our scope
regions = ["Auvergne-Rhône-Alpes", "Île-de-France", "Nouvelle-Aquitaine", "Occitanie","Provence-Alpes-Côte d'Azur"]

Next, we define our service

In [ ]:
@bentoml.service()
class TransactionPredictionRegion:
    

    def __init__(self):
        transaction_classifiers = { region :
            bentoml.models.get(
        "transaction_below_market_identifier_{region}:latest"
    ) for region in list_regions
        }
        transaction_regressors = { region :
            bentoml.models.get(
        "transaction_value_estimator_{region}:latest"
    ) for region in list_regions
        }

    @bentoml.api
    def predict(self, region, transactions: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        
        self.classifier = bentoml.sklearn.load_model(self.transaction_classifiers[region])
        self.regressor = bentoml.sklearn.load_model(self.transaction_regressors[region])
        predicted_classes = self.classifier.predict(transactions)
        predicted_transactions = self.regressor.predict(transactions)
        return (predicted_transactions, predicted_classes)


Two approaches (among others) are possible:
* As soon as the service is started (with the command bentoml serve), all models (5 classification models and 5 regression models) are loaded upfront. This is what we have done here.
* We load only the 2 relevant models at the moment a user calls the API for inference.

There is no universally correct choice — it all depends on the project constraints (desired API response time, traffic volume the API needs to handle, etc.).

Here, we went with the first approach, as the models are relatively lightweight and we have few regions.

A more scalable approach would be to have multiple active instances of the same API, each serving one region.


Given the endpoint definition, the expected input is a region and a set of observations in the form of a numpy array. As seen in the course, we can:
* Use the terminal with curl
* Use the Swagger UI
* Use Python, with the requests library directly or indirectly via BentoML (as done below)


In [ ]:
'''
After running the bentoml serve command, we can interact with
the model by specifying the region and the transactions to predict.
'''

with bentoml.SyncHTTPClient("http://localhost:3000") as client:
    result = client.predict(region = "Occitanie", transactions=test_values)
